# 03. Sentinel-1 radar features and spatial forward feature selection

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from model_aza.pilots import climate, clustering, data, earth_engine, evaluation, features
from model_aza.training import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
subset_run = ""  # Optional pipeline_outputs/pilots/subset_<id>; blank uses the latest
subset_run = Path(subset_run) if subset_run else tuning.latest_run(root, "subset", area=AREA)
rows, arrays = data.load(subset_run)
cube = features.stack(arrays["s2_values"])
valid = arrays["s2_valid"]
print("Subset:", subset_run, "| pixels:", len(rows))

## Sentinel-1 coverage

In [ ]:
s1 = earth_engine.s1_features_cube(arrays["s1_values"])
s1_valid = np.isfinite(s1).all(axis=2)
print("Mean Sentinel-1 half-months per pixel:", round(float(s1_valid.sum(axis=1).mean()), 1))
display(pd.DataFrame({"partition": [r["partition"] for r in rows], "s1_half_months": s1_valid.sum(axis=1)})
        .groupby("partition").s1_half_months.describe().round(1))

## Feature sets

In [ ]:
s2_X, s2_ok, s2_cols = features.harmonic_features(cube, valid)
s1_X, s1_ok, s1_cols = features.harmonic_features(s1, s1_valid, names=("VV", "VH", "VHVV"))
both_X, both_ok, both_cols = np.hstack([s2_X, s1_X]), s2_ok & s1_ok, s2_cols + s1_cols
sets = {"s2_harmonic": (s2_X, s2_ok), "s1_harmonic": (s1_X, s1_ok), "s2_plus_s1_harmonic": (both_X, both_ok)}
pd.DataFrame([{"feature_set": k, "predictors": X.shape[1], "pixels": int(ok.sum())} for k, (X, ok) in sets.items()])

## Evaluate on pixels usable by every set

In [ ]:
run = tuning.new_run(root, "sentinel1", subset_run / "meta.json", area=AREA)
common = s2_ok & s1_ok
summaries, details = [], []
for name, (X, _) in sets.items():
    s, d = repeated(name, np.nan_to_num(X), rows, common)
    summaries += s
    details += d
    print(name, "done", flush=True)
tuning.save_records(run, "summary", summaries)
tuning.save_records(run, "dataset_holdout", details)
display(summarize(summaries))

## Spatial forward feature selection over variable groups

In [ ]:
groups = features.column_groups(both_cols)
fast = {**evaluation.PARAMS, "n_estimators": 150}
selected, history = evaluation.forward_group_selection(np.nan_to_num(both_X), rows, common, groups, fast,
                                                       progress=lambda m: print(m, flush=True))
tuning.save_records(run, "selection_history", history)
display(pd.DataFrame(history).round(3))
print("Selected groups:", selected)

## Selected groups

In [ ]:
columns = [c for name in selected for c in groups[name]]
s, d = repeated("selected_groups", np.nan_to_num(both_X[:, columns]), rows, common)
summaries += s
tuning.save_records(run, "summary_with_selection", summaries)
display(summarize(summaries))

## Figure

In [ ]:
table = pd.DataFrame(summaries).groupby("feature_set", sort=False)[
    ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1"]].mean()
figure, axis = plt.subplots(figsize=(10, 4))
table.T.plot(kind="bar", ax=axis)
axis.set_ylim(0, 1)
axis.set_ylabel("Parcel macro-F1 (mean of seeds)")
show_and_save(figure, run, "sentinel1_selection")
print("Run folder:", run)